## PHASE 6.4 — Pathway-level Features  
## Step 6.4.1: Pathway annotation loading


In [10]:
import pandas as pd

reactome = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\NCBI2Reactome_All_Levels.txt",
    sep="\t",
    header=None,
    engine="python",      # critical
    on_bad_lines="skip"   # safely skip malformed rows
)

reactome.head(), reactome.shape


(   0              1                                                  2  \
 0  1   R-HSA-109582  https://reactome.org/PathwayBrowser/#/R-HSA-10...   
 1  1   R-HSA-114608  https://reactome.org/PathwayBrowser/#/R-HSA-11...   
 2  1   R-HSA-168249  https://reactome.org/PathwayBrowser/#/R-HSA-16...   
 3  1   R-HSA-168256  https://reactome.org/PathwayBrowser/#/R-HSA-16...   
 4  1  R-HSA-6798695  https://reactome.org/PathwayBrowser/#/R-HSA-67...   
 
                           3    4             5  
 0                Hemostasis  TAS  Homo sapiens  
 1   Platelet degranulation   TAS  Homo sapiens  
 2      Innate Immune System  TAS  Homo sapiens  
 3             Immune System  TAS  Homo sapiens  
 4  Neutrophil degranulation  TAS  Homo sapiens  ,
 (1369380, 6))

In [12]:
# select correct columns
reactome_clean = reactome[[0, 1, 3, 5]].copy()
reactome_clean.columns = [
    "entrez_id",
    "reactome_id",
    "pathway_name",
    "species"
]

# keep human only
reactome_clean = reactome_clean[
    reactome_clean["species"] == "Homo sapiens"
]

# drop species column (no longer needed)
reactome_clean = reactome_clean.drop(columns=["species"])

reactome_clean.head(), reactome_clean.shape


(  entrez_id    reactome_id              pathway_name
 0         1   R-HSA-109582                Hemostasis
 1         1   R-HSA-114608   Platelet degranulation 
 2         1   R-HSA-168249      Innate Immune System
 3         1   R-HSA-168256             Immune System
 4         1  R-HSA-6798695  Neutrophil degranulation,
 (283499, 3))

In [14]:
reactome_clean["entrez_id"].value_counts().head()


entrez_id
6233    1014
7311    1002
7314     946
7316     946
2885     476
Name: count, dtype: int64

In [16]:
# If not installed before, install once:
# pip install mygene

import mygene
import pandas as pd

mg = mygene.MyGeneInfo()

# get unique Entrez IDs from Reactome
entrez_ids = reactome_clean["entrez_id"].astype(str).unique().tolist()

# query mygene (batch query)
res = mg.querymany(
    entrez_ids,
    scopes="entrezgene",
    fields="symbol",
    species="human",
    as_dataframe=True
)

# keep successful mappings
entrez_to_symbol = (
    res[["symbol"]]
    .dropna()
    .reset_index()
    .rename(columns={"index": "entrez_id"})
)

entrez_to_symbol.head(), entrez_to_symbol.shape


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
372 input query terms found no hit:	['24255210', '1238024', '1238048', '1253480', '1254294', '1254395', '13229470', '13229472', '1322947


(   query symbol
 0      1   A1BG
 1     10   NAT2
 2    100    ADA
 3   1000   CDH2
 4  10000   AKT3,
 (11230, 2))

In [19]:
## PHASE 6.4.3 — Build HGNC-based gene → pathway mapping

In [18]:
# ensure consistent dtypes
reactome_clean["entrez_id"] = reactome_clean["entrez_id"].astype(str)
entrez_to_symbol["query"] = entrez_to_symbol["query"].astype(str)

# merge to attach HGNC symbols
reactome_hgnc = reactome_clean.merge(
    entrez_to_symbol,
    left_on="entrez_id",
    right_on="query",
    how="inner"
)

# keep only relevant columns
reactome_hgnc = reactome_hgnc[
    ["symbol", "reactome_id", "pathway_name"]
].drop_duplicates()

reactome_hgnc.head(), reactome_hgnc.shape


(  symbol    reactome_id              pathway_name
 0   A1BG   R-HSA-109582                Hemostasis
 1   A1BG   R-HSA-114608   Platelet degranulation 
 2   A1BG   R-HSA-168249      Innate Immune System
 3   A1BG   R-HSA-168256             Immune System
 4   A1BG  R-HSA-6798695  Neutrophil degranulation,
 (134814, 3))

In [23]:
# Build pathway sets for each drug

In [20]:
import pandas as pd

drug_targets = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

drug_targets.head(), drug_targets.shape, drug_targets.columns


(  target_chembl_id    targets targets_ppi
 0       CHEMBL1778  ['IL2RA']   ['IL2RA']
 1       CHEMBL1782   ['FDPS']    ['FDPS']
 2       CHEMBL1783  ['VEGFA']   ['VEGFA']
 3       CHEMBL1785  ['EDNRB']   ['EDNRB']
 4       CHEMBL1786  ['IMPA1']   ['IMPA1'],
 (522, 3),
 Index(['target_chembl_id', 'targets', 'targets_ppi'], dtype='object'))

In [22]:
import ast
import pandas as pd

# make a copy
drug_targets_exp = drug_targets.copy()

# convert string list to real list
drug_targets_exp["targets_ppi"] = drug_targets_exp["targets_ppi"].apply(ast.literal_eval)

# explode so one gene per row
drug_targets_exp = drug_targets_exp.explode("targets_ppi")

# rename for clarity
drug_targets_exp = drug_targets_exp.rename(
    columns={"target_chembl_id": "drug", "targets_ppi": "gene"}
)

drug_targets_exp.head(), drug_targets_exp.shape


(         drug    targets   gene
 0  CHEMBL1778  ['IL2RA']  IL2RA
 1  CHEMBL1782   ['FDPS']   FDPS
 2  CHEMBL1783  ['VEGFA']  VEGFA
 3  CHEMBL1785  ['EDNRB']  EDNRB
 4  CHEMBL1786  ['IMPA1']  IMPA1,
 (522, 3))

In [24]:
drug_pathways = drug_targets_exp.merge(
    reactome_hgnc,
    left_on="gene",
    right_on="symbol",
    how="inner"
)

drug_pathways = drug_pathways[
    ["drug", "reactome_id", "pathway_name"]
].drop_duplicates()

drug_pathways.head(), drug_pathways.shape


(         drug    reactome_id                         pathway_name
 0  CHEMBL1778  R-HSA-1280215  Cytokine Signaling in Immune system
 1  CHEMBL1778   R-HSA-162582                  Signal Transduction
 2  CHEMBL1778   R-HSA-168256                        Immune System
 3  CHEMBL1778   R-HSA-212436        Generic Transcription Pathway
 4  CHEMBL1778   R-HSA-449147            Signaling by Interleukins,
 (7901, 3))

In [26]:
# assuming this variable already exists in memory
# name may be drug_pathways or similar

drug_pathways.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv",
    index=False
)

drug_pathways.shape


(7901, 3)

In [33]:
# PHASE 6.4.5 — Pathway sets for top PageRank genes (per subtype)


In [28]:
import pandas as pd

pr_luma = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_PPI_Pagerank.csv"
)

pr_luma.head(), pr_luma.shape, pr_luma.columns


(  Unnamed: 0  pagerank
 0       TP53  0.002233
 1        SRC  0.002190
 2      EP300  0.001869
 3       EGFR  0.001740
 4       HRAS  0.001672,
 (9875, 2),
 Index(['Unnamed: 0', 'pagerank'], dtype='object'))

In [30]:
# rename gene column properly
pr_luma = pr_luma.rename(columns={"Unnamed: 0": "gene"})

pr_luma.head(), pr_luma.columns


(    gene  pagerank
 0   TP53  0.002233
 1    SRC  0.002190
 2  EP300  0.001869
 3   EGFR  0.001740
 4   HRAS  0.001672,
 Index(['gene', 'pagerank'], dtype='object'))

In [32]:
# select top 5% PageRank genes
top_n = int(0.05 * pr_luma.shape[0])

top_luma_genes = set(
    pr_luma.sort_values("pagerank", ascending=False)
           .head(top_n)["gene"]
)

len(top_luma_genes)


493

In [34]:
# subset Reactome annotations to LumA core network genes
luma_pathways = reactome_hgnc[
    reactome_hgnc["symbol"].isin(top_luma_genes)
]

luma_pathways.shape, luma_pathways.head()


((14229, 3),
    symbol    reactome_id                                       pathway_name
 24   CDH2  R-HSA-1266738                              Developmental Biology
 26   CDH2  R-HSA-1500931                            Cell-Cell communication
 27   CDH2   R-HSA-381426  Regulation of Insulin-like Growth Factor (IGF)...
 28   CDH2   R-HSA-392499                             Metabolism of proteins
 29   CDH2   R-HSA-418990                    Adherens junctions interactions)

In [36]:
luma_pathway_scores = (
    luma_pathways
    .merge(pr_luma, left_on="symbol", right_on="gene")
    .groupby(["reactome_id", "pathway_name"])["pagerank"]
    .sum()
    .reset_index()
    .rename(columns={"pagerank": "luma_pathway_score"})
    .sort_values("luma_pathway_score", ascending=False)
)

luma_pathway_scores.head(), luma_pathway_scores.shape


(        reactome_id                     pathway_name  luma_pathway_score
 134    R-HSA-162582              Signal Transduction            0.153784
 153   R-HSA-1643685                          Disease            0.126282
 183    R-HSA-168256                    Immune System            0.109753
 72    R-HSA-1266738            Developmental Biology            0.097819
 1090    R-HSA-74160  Gene expression (Transcription)            0.086774,
 (1775, 3))

In [38]:
# save the CORRECT LumA pathway scores
luma_pathway_scores.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_Pathway_Scores.csv",
    index=False
)

print("✅ Correct LumA_Pathway_Scores.csv saved")


✅ Correct LumA_Pathway_Scores.csv saved


In [40]:
import pandas as pd

# 1️⃣ Load drug → pathway mapping
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

# 2️⃣ Load Luminal A pathway scores
luma_pathway_scores = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_Pathway_Scores.csv"
)

# quick sanity check
drug_pathways.shape, luma_pathway_scores.shape


((7901, 3), (1775, 3))

In [42]:
# merge drug → pathway with Luminal A pathway scores
drug_luma_pathways = drug_pathways.merge(
    luma_pathway_scores,
    on=["reactome_id", "pathway_name"],
    how="inner"
)

# sanity check
drug_luma_pathways.shape, drug_luma_pathways.head()


((7225, 4),
          drug    reactome_id                         pathway_name  \
 0  CHEMBL1778  R-HSA-1280215  Cytokine Signaling in Immune system   
 1  CHEMBL1778   R-HSA-162582                  Signal Transduction   
 2  CHEMBL1778   R-HSA-168256                        Immune System   
 3  CHEMBL1778   R-HSA-212436        Generic Transcription Pathway   
 4  CHEMBL1778   R-HSA-449147            Signaling by Interleukins   
 
    luma_pathway_score  
 0            0.080565  
 1            0.153784  
 2            0.109753  
 3            0.081979  
 4            0.064782  )

In [44]:
# recreate merged table (explicit, notebook-safe)
drug_pathway_luma = drug_pathways.merge(
    luma_pathway_scores,
    on=["reactome_id", "pathway_name"],
    how="inner"
)

# aggregate pathway scores per drug
drug_pathway_luma_agg = (
    drug_pathway_luma
    .groupby(["drug", "reactome_id", "pathway_name"], as_index=False)
    .agg(
        luma_pathway_score_sum=("luma_pathway_score", "sum"),
        n_targets=("luma_pathway_score", "count")
    )
)

drug_pathway_luma_agg.shape, drug_pathway_luma_agg.head()


((7225, 5),
          drug    reactome_id                         pathway_name  \
 0  CHEMBL1778  R-HSA-1280215  Cytokine Signaling in Immune system   
 1  CHEMBL1778   R-HSA-162582                  Signal Transduction   
 2  CHEMBL1778   R-HSA-168256                        Immune System   
 3  CHEMBL1778   R-HSA-212436        Generic Transcription Pathway   
 4  CHEMBL1778   R-HSA-449147            Signaling by Interleukins   
 
    luma_pathway_score_sum  n_targets  
 0                0.080565          1  
 1                0.153784          1  
 2                0.109753          1  
 3                0.081979          1  
 4                0.064782          1  )

In [46]:
# aggregate to drug-level features (Luminal A)
drug_features_luma = (
    drug_pathway_luma_agg
    .groupby("drug", as_index=False)
    .agg(
        n_pathways_luma=("reactome_id", "nunique"),
        mean_pathway_score_luma=("luma_pathway_score_sum", "mean"),
        max_pathway_score_luma=("luma_pathway_score_sum", "max"),
        sum_pathway_score_luma=("luma_pathway_score_sum", "sum")
    )
)

drug_features_luma.shape, drug_features_luma.head()


((510, 5),
          drug  n_pathways_luma  mean_pathway_score_luma  \
 0  CHEMBL1778               16                 0.054038   
 1  CHEMBL1782                5                 0.015057   
 2  CHEMBL1783               29                 0.046635   
 3  CHEMBL1785               10                 0.041135   
 4  CHEMBL1786                2                 0.023320   
 
    max_pathway_score_luma  sum_pathway_score_luma  
 0                0.153784                0.864616  
 1                0.045174                0.075286  
 2                0.153784                1.352424  
 3                0.153784                0.411352  
 4                0.045174                0.046641  )

In [48]:
drug_features_luma.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumA.csv",
    index=False
)


In [57]:
## ▶️ Phase 6.4.2 — LumB Pathway-level Scoring


In [50]:
# save Reactome ↔ HGNC mapping for reuse
reactome_hgnc.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Reactome_HGNC_Mapping.csv",
    index=False
)

reactome_hgnc.shape


(134814, 3)

In [52]:
# Phase 6.4.3 — Luminal B
# Step 1: Load Luminal B PageRank

import pandas as pd

pr_lumb = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_PPI_Pagerank.csv"
)

# fix gene column if needed
pr_lumb = pr_lumb.rename(columns={"Unnamed: 0": "gene"})

pr_lumb.head(), pr_lumb.shape


(     gene  pagerank
 0     SRC  0.001973
 1    TP53  0.001848
 2    EGFR  0.001528
 3   EP300  0.001395
 4  RPS27A  0.001390,
 (9875, 2))

In [54]:
# select top 5% PageRank genes for Luminal B
top_n = int(0.05 * pr_lumb.shape[0])

top_lumb_genes = set(
    pr_lumb.sort_values("pagerank", ascending=False)
           .head(top_n)["gene"]
)

len(top_lumb_genes)


493

In [56]:
# subset Reactome annotations to Luminal B core network genes
lumb_pathways = reactome_hgnc[
    reactome_hgnc["symbol"].isin(top_lumb_genes)
]

lumb_pathways.shape, lumb_pathways.head()


((14871, 3),
    symbol    reactome_id                                       pathway_name
 24   CDH2  R-HSA-1266738                              Developmental Biology
 26   CDH2  R-HSA-1500931                            Cell-Cell communication
 27   CDH2   R-HSA-381426  Regulation of Insulin-like Growth Factor (IGF)...
 28   CDH2   R-HSA-392499                             Metabolism of proteins
 29   CDH2   R-HSA-418990                    Adherens junctions interactions)

In [58]:
# merge LumB core genes with their PageRank scores
lumb_gene_scores = (
    pr_lumb.merge(
        lumb_pathways,
        left_on="gene",
        right_on="symbol",
        how="inner"
    )
)

# aggregate gene → pathway scores
lumb_pathway_scores = (
    lumb_gene_scores
    .groupby(["reactome_id", "pathway_name"], as_index=False)
    .agg(
        pagerank_sum=("pagerank", "sum"),
        n_genes=("gene", "nunique")
    )
)

# normalize pathway score
lumb_pathway_scores["lumb_pathway_score"] = (
    lumb_pathway_scores["pagerank_sum"] /
    lumb_pathway_scores["n_genes"]
)

lumb_pathway_scores.shape, lumb_pathway_scores.head()


((1759, 5),
      reactome_id                     pathway_name  pagerank_sum  n_genes  \
 0  R-HSA-1059683          Interleukin-6 signaling      0.004971        7   
 1   R-HSA-109581                        Apoptosis      0.007990       13   
 2   R-HSA-109582                       Hemostasis      0.057234      104   
 3   R-HSA-109606  Intrinsic Pathway for Apoptosis      0.004732        8   
 4   R-HSA-109704                     PI3K Cascade      0.005661        9   
 
    lumb_pathway_score  
 0            0.000710  
 1            0.000615  
 2            0.000550  
 3            0.000591  
 4            0.000629  )

In [60]:
# save the CORRECT LumA pathway scores
lumb_pathway_scores.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_Pathway_Scores.csv",
    index=False
)

print("✅ Correct LumB_Pathway_Scores.csv saved")


✅ Correct LumB_Pathway_Scores.csv saved


In [145]:
# merge drug → pathway with Luminal B pathway scores
drug_lumb_pathways = drug_pathways.merge(
    lumb_pathway_scores,
    on=["reactome_id", "pathway_name"],
    how="inner"
)

# sanity check
drug_lumb_pathways.shape, drug_lumb_pathways.head()


((7316, 6),
          drug    reactome_id                         pathway_name  \
 0  CHEMBL1778  R-HSA-1280215  Cytokine Signaling in Immune system   
 1  CHEMBL1778   R-HSA-162582                  Signal Transduction   
 2  CHEMBL1778   R-HSA-168256                        Immune System   
 3  CHEMBL1778   R-HSA-212436        Generic Transcription Pathway   
 4  CHEMBL1778   R-HSA-449147            Signaling by Interleukins   
 
    pagerank_sum  n_genes  lumb_pathway_score  
 0      0.066205      117            0.000566  
 1      0.133821      273            0.000490  
 2      0.093894      183            0.000513  
 3      0.064268      121            0.000531  
 4      0.052615       93            0.000566  )

In [147]:
# aggregate to drug-level features (Luminal B)
drug_features_lumb = (
    drug_lumb_pathways
    .groupby("drug", as_index=False)
    .agg(
        n_pathways_lumb=("reactome_id", "nunique"),
        mean_pathway_score_lumb=("lumb_pathway_score", "mean"),
        max_pathway_score_lumb=("lumb_pathway_score", "max"),
        sum_pathway_score_lumb=("lumb_pathway_score", "sum")
    )
)

drug_features_lumb.shape, drug_features_lumb.head()


((510, 5),
          drug  n_pathways_lumb  mean_pathway_score_lumb  \
 0  CHEMBL1778               16                 0.000553   
 1  CHEMBL1782                6                 0.000444   
 2  CHEMBL1783               29                 0.000612   
 3  CHEMBL1785               10                 0.000487   
 4  CHEMBL1786                2                 0.000492   
 
    max_pathway_score_lumb  sum_pathway_score_lumb  
 0                0.000623                0.008854  
 1                0.000529                0.002664  
 2                0.001022                0.017737  
 3                0.000611                0.004873  
 4                0.000494                0.000983  )

In [149]:
drug_features_lumb.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumB.csv",
    index=False
)


In [85]:
# load HER2 PageRank
pr_her2 = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_PPI_Pagerank.csv"
)

# fix gene column if needed
pr_her2 = pr_her2.rename(columns={"Unnamed: 0": "gene"})

# select top 5% PageRank genes
top_n = int(0.05 * pr_her2.shape[0])
top_her2_genes = set(
    pr_her2.sort_values("pagerank", ascending=False)
           .head(top_n)["gene"]
)

pr_her2.head(), pr_her2.shape, len(top_her2_genes)


(     gene  pagerank
 0     SRC  0.001975
 1    TP53  0.001818
 2    EGFR  0.001521
 3  RPS27A  0.001392
 4   EP300  0.001352,
 (9875, 2),
 493)

In [87]:
# subset Reactome annotations to HER2 core network genes
her2_pathways = reactome_hgnc[
    reactome_hgnc["symbol"].isin(top_her2_genes)
]

her2_pathways.shape, her2_pathways.head()


((14762, 3),
    symbol    reactome_id                                       pathway_name
 24   CDH2  R-HSA-1266738                              Developmental Biology
 26   CDH2  R-HSA-1500931                            Cell-Cell communication
 27   CDH2   R-HSA-381426  Regulation of Insulin-like Growth Factor (IGF)...
 28   CDH2   R-HSA-392499                             Metabolism of proteins
 29   CDH2   R-HSA-418990                    Adherens junctions interactions)

In [89]:
# merge HER2 core genes with their PageRank scores
her2_gene_scores = (
    pr_her2.merge(
        her2_pathways,
        left_on="gene",
        right_on="symbol",
        how="inner"
    )
)

# aggregate gene → pathway scores
her2_pathway_scores = (
    her2_gene_scores
    .groupby(["reactome_id", "pathway_name"], as_index=False)
    .agg(
        pagerank_sum=("pagerank", "sum"),
        n_genes=("gene", "nunique")
    )
)

# normalize pathway score
her2_pathway_scores["her2_pathway_score"] = (
    her2_pathway_scores["pagerank_sum"] /
    her2_pathway_scores["n_genes"]
)

her2_pathway_scores.shape, her2_pathway_scores.head()


((1774, 5),
      reactome_id                     pathway_name  pagerank_sum  n_genes  \
 0  R-HSA-1059683          Interleukin-6 signaling      0.004783        7   
 1   R-HSA-109581                        Apoptosis      0.007840       13   
 2   R-HSA-109582                       Hemostasis      0.056104      103   
 3   R-HSA-109606  Intrinsic Pathway for Apoptosis      0.004644        8   
 4   R-HSA-109704                     PI3K Cascade      0.005247        8   
 
    her2_pathway_score  
 0            0.000683  
 1            0.000603  
 2            0.000545  
 3            0.000580  
 4            0.000656  )

In [91]:
# save the CORRECT LumA pathway scores
her2_pathway_scores.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_Pathway_Scores.csv",
    index=False
)

print("✅ Correct HER2_Pathway_Scores.csv saved")


✅ Correct HER2_Pathway_Scores.csv saved


In [157]:
# load drug → pathway mapping
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

# merge with HER2 pathway scores
drug_her2_pathways = drug_pathways.merge(
    her2_pathway_scores[["reactome_id", "pathway_name", "her2_pathway_score"]],
    on=["reactome_id", "pathway_name"],
    how="inner"
)

drug_her2_pathways.shape, drug_her2_pathways.head()


((7321, 4),
          drug    reactome_id                         pathway_name  \
 0  CHEMBL1778  R-HSA-1280215  Cytokine Signaling in Immune system   
 1  CHEMBL1778   R-HSA-162582                  Signal Transduction   
 2  CHEMBL1778   R-HSA-168256                        Immune System   
 3  CHEMBL1778   R-HSA-212436        Generic Transcription Pathway   
 4  CHEMBL1778   R-HSA-449147            Signaling by Interleukins   
 
    her2_pathway_score  
 0            0.000554  
 1            0.000486  
 2            0.000501  
 3            0.000528  
 4            0.000553  )

In [159]:
# aggregate pathway scores per drug (HER2)
drug_pathway_her2_agg = (
    drug_her2_pathways
    .groupby(["drug", "reactome_id", "pathway_name"], as_index=False)
    .agg(
        her2_pathway_score_sum=("her2_pathway_score", "sum"),
        n_targets=("her2_pathway_score", "count")
    )
)

drug_pathway_her2_agg.shape, drug_pathway_her2_agg.head()


((7321, 5),
          drug    reactome_id                         pathway_name  \
 0  CHEMBL1778  R-HSA-1280215  Cytokine Signaling in Immune system   
 1  CHEMBL1778   R-HSA-162582                  Signal Transduction   
 2  CHEMBL1778   R-HSA-168256                        Immune System   
 3  CHEMBL1778   R-HSA-212436        Generic Transcription Pathway   
 4  CHEMBL1778   R-HSA-449147            Signaling by Interleukins   
 
    her2_pathway_score_sum  n_targets  
 0                0.000554          1  
 1                0.000486          1  
 2                0.000501          1  
 3                0.000528          1  
 4                0.000553          1  )

In [161]:
# aggregate to drug-level features (HER2)
drug_features_her2 = (
    drug_pathway_her2_agg
    .groupby("drug", as_index=False)
    .agg(
        n_pathways_her2=("reactome_id", "nunique"),
        mean_pathway_score_her2=("her2_pathway_score_sum", "mean"),
        max_pathway_score_her2=("her2_pathway_score_sum", "max"),
        sum_pathway_score_her2=("her2_pathway_score_sum", "sum")
    )
)

drug_features_her2.shape, drug_features_her2.head()


((510, 5),
          drug  n_pathways_her2  mean_pathway_score_her2  \
 0  CHEMBL1778               16                 0.000545   
 1  CHEMBL1782                5                 0.000450   
 2  CHEMBL1783               29                 0.000606   
 3  CHEMBL1785               10                 0.000481   
 4  CHEMBL1786                2                 0.000490   
 
    max_pathway_score_her2  sum_pathway_score_her2  
 0                0.000627                0.008726  
 1                0.000511                0.002250  
 2                0.001017                0.017570  
 3                0.000591                0.004814  
 4                0.000497                0.000979  )

In [163]:
drug_features_her2.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_HER2.csv",
    index=False
)


In [120]:
import pandas as pd

# load TNBC PageRank
pr_tnbc = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TNBC_PPI_Pagerank.csv"
)

# rename gene column if needed
if "Unnamed: 0" in pr_tnbc.columns:
    pr_tnbc = pr_tnbc.rename(columns={"Unnamed: 0": "gene"})

pr_tnbc.head(), pr_tnbc.shape


(     gene  pagerank
 0     SRC  0.001931
 1    TP53  0.001838
 2    EGFR  0.001459
 3  RPS27A  0.001389
 4   EP300  0.001337,
 (9875, 2))

In [122]:
# select top 5% TNBC network genes
top_n = int(0.05 * pr_tnbc.shape[0])

top_tnbc_genes = set(
    pr_tnbc.sort_values("pagerank", ascending=False)
           .head(top_n)["gene"]
)

len(top_tnbc_genes)


493

In [124]:
# subset Reactome annotations to TNBC core network genes
tnbc_pathways = reactome_hgnc[
    reactome_hgnc["symbol"].isin(top_tnbc_genes)
]

tnbc_pathways.shape, tnbc_pathways.head()


((14606, 3),
    symbol    reactome_id                                       pathway_name
 24   CDH2  R-HSA-1266738                              Developmental Biology
 26   CDH2  R-HSA-1500931                            Cell-Cell communication
 27   CDH2   R-HSA-381426  Regulation of Insulin-like Growth Factor (IGF)...
 28   CDH2   R-HSA-392499                             Metabolism of proteins
 29   CDH2   R-HSA-418990                    Adherens junctions interactions)

In [126]:
# merge TNBC core genes with PageRank scores
tnbc_pr = pr_tnbc.merge(
    tnbc_pathways,
    left_on="gene",
    right_on="symbol",
    how="inner"
)

# aggregate to pathway-level scores
tnbc_pathway_scores = (
    tnbc_pr
    .groupby(["reactome_id", "pathway_name"], as_index=False)
    .agg(
        pagerank_sum=("pagerank", "sum"),
        n_genes=("gene", "nunique")
    )
)

# normalize (mean PageRank per gene in pathway)
tnbc_pathway_scores["tnbc_pathway_score"] = (
    tnbc_pathway_scores["pagerank_sum"] / tnbc_pathway_scores["n_genes"]
)

tnbc_pathway_scores.shape, tnbc_pathway_scores.head()


((1776, 5),
      reactome_id                     pathway_name  pagerank_sum  n_genes  \
 0  R-HSA-1059683          Interleukin-6 signaling      0.004363        6   
 1   R-HSA-109581                        Apoptosis      0.007954       13   
 2   R-HSA-109582                       Hemostasis      0.056075      105   
 3   R-HSA-109606  Intrinsic Pathway for Apoptosis      0.004717        8   
 4   R-HSA-109704                     PI3K Cascade      0.005464        9   
 
    tnbc_pathway_score  
 0            0.000727  
 1            0.000612  
 2            0.000534  
 3            0.000590  
 4            0.000607  )

In [128]:
# save the CORRECT LumA pathway scores
tnbc_pathway_scores.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TNBC_Pathway_Scores.csv",
    index=False
)

print("✅ Correct TNBC_Pathway_Scores.csv saved")


✅ Correct TNBC_Pathway_Scores.csv saved


In [173]:
# merge drug → pathway with TNBC pathway scores
drug_tnbc_pathways = drug_pathways.merge(
    tnbc_pathway_scores,
    on=["reactome_id", "pathway_name"],
    how="inner"
)

drug_tnbc_pathways.shape, drug_tnbc_pathways.head()


((7327, 6),
          drug    reactome_id                         pathway_name  \
 0  CHEMBL1778  R-HSA-1280215  Cytokine Signaling in Immune system   
 1  CHEMBL1778   R-HSA-162582                  Signal Transduction   
 2  CHEMBL1778   R-HSA-168256                        Immune System   
 3  CHEMBL1778   R-HSA-212436        Generic Transcription Pathway   
 4  CHEMBL1778   R-HSA-449147            Signaling by Interleukins   
 
    pagerank_sum  n_genes  tnbc_pathway_score  
 0      0.063752      115            0.000554  
 1      0.128967      267            0.000483  
 2      0.091145      181            0.000504  
 3      0.061821      116            0.000533  
 4      0.050487       91            0.000555  )

In [175]:
# aggregate pathway scores per drug (TNBC)
drug_features_tnbc = (
    drug_tnbc_pathways
    .groupby("drug", as_index=False)
    .agg(
        n_pathways_tnbc=("reactome_id", "nunique"),
        mean_pathway_score_tnbc=("tnbc_pathway_score", "mean"),
        max_pathway_score_tnbc=("tnbc_pathway_score", "max"),
        sum_pathway_score_tnbc=("tnbc_pathway_score", "sum")
    )
)

drug_features_tnbc.shape, drug_features_tnbc.head()


((510, 5),
          drug  n_pathways_tnbc  mean_pathway_score_tnbc  \
 0  CHEMBL1778               16                 0.000544   
 1  CHEMBL1782                5                 0.000443   
 2  CHEMBL1783               29                 0.000600   
 3  CHEMBL1785               10                 0.000482   
 4  CHEMBL1786                2                 0.000479   
 
    max_pathway_score_tnbc  sum_pathway_score_tnbc  
 0                0.000618                0.008698  
 1                0.000504                0.002214  
 2                0.001012                0.017412  
 3                0.000594                0.004819  
 4                0.000479                0.000957  )

In [177]:
# save TNBC drug–pathway features
drug_features_tnbc.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_TNBC.csv",
    index=False
)

print("✅ Saved: Drug_Pathway_Features_TNBC.csv")


✅ Saved: Drug_Pathway_Features_TNBC.csv


In [181]:
## Phase 6.4.3 — Combine pathway features across subtypes


In [148]:
import pandas as pd

# load subtype-specific pathway features
luma = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumA.csv"
)
lumb = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumB.csv"
)
her2 = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_HER2.csv"
)
tnbc = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_TNBC.csv"
)

# merge step-by-step on drug
features_all = (
    luma
    .merge(lumb, on="drug", how="outer")
    .merge(her2, on="drug", how="outer")
    .merge(tnbc, on="drug", how="outer")
)

features_all.shape, features_all.head()


((510, 17),
          drug  n_pathways_luma  mean_pathway_score_luma  \
 0  CHEMBL1778               16                 0.054038   
 1  CHEMBL1782                5                 0.015057   
 2  CHEMBL1783               29                 0.046635   
 3  CHEMBL1785               10                 0.041135   
 4  CHEMBL1786                2                 0.023320   
 
    max_pathway_score_luma  sum_pathway_score_luma  n_pathways_lumb  \
 0                0.153784                0.864616               16   
 1                0.045174                0.075286                6   
 2                0.153784                1.352424               29   
 3                0.153784                0.411352               10   
 4                0.045174                0.046641                2   
 
    mean_pathway_score_lumb  max_pathway_score_lumb  sum_pathway_score_lumb  \
 0                 0.000553                0.000623                0.008854   
 1                 0.000444             

In [150]:
# save the CORRECT LumA pathway scores
features_all.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\features_all.csv",
    index=False
)

print("✅ Correct features_all.csv saved")


✅ Correct features_all.csv saved


# Pathway overlap score

In [186]:
import pandas as pd

# 1) load inputs
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

luma_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_Pathway_Scores.csv"
)

# 2) create sets
drug_to_pathways = (
    drug_pathways
    .groupby("drug")["reactome_id"]
    .apply(set)
)

luma_core_pathways = set(luma_pathways["reactome_id"])

# 3) Jaccard overlap
overlap_records = []

for drug, d_paths in drug_to_pathways.items():
    intersection = len(d_paths & luma_core_pathways)
    union = len(d_paths | luma_core_pathways)
    jaccard = intersection / union if union > 0 else 0
    
    overlap_records.append({
        "drug": drug,
        "luma_pathway_overlap_jaccard": jaccard,
        "n_shared_pathways_luma": intersection
    })

drug_overlap_luma = pd.DataFrame(overlap_records)

# 4) inspect
drug_overlap_luma.shape, drug_overlap_luma.head()


((512, 3),
          drug  luma_pathway_overlap_jaccard  n_shared_pathways_luma
 0  CHEMBL1778                      0.006006                      16
 1  CHEMBL1782                      0.002628                       7
 2  CHEMBL1783                      0.010886                      29
 3  CHEMBL1785                      0.003754                      10
 4  CHEMBL1786                      0.001126                       3)

In [188]:
drug_overlap_luma.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_LumA.csv",
    index=False
)

print("✅ Saved: Drug_Pathway_Overlap_LumA.csv")


✅ Saved: Drug_Pathway_Overlap_LumA.csv


In [190]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# load drug–pathway scores (LumA)
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

luma_pathway_scores = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_Pathway_Scores.csv"
)

# pivot to drug × pathway matrix
drug_pathway_matrix = (
    drug_pathways
    .merge(luma_pathway_scores, on=["reactome_id", "pathway_name"], how="inner")
    .pivot_table(
        index="drug",
        columns="reactome_id",
        values="luma_pathway_score",
        fill_value=0
    )
)

# LumA reference vector (mean pathway activity)
luma_ref = (
    luma_pathway_scores
    .groupby("reactome_id")["luma_pathway_score"]
    .mean()
    .reindex(drug_pathway_matrix.columns, fill_value=0)
    .values
    .reshape(1, -1)
)

# cosine similarity
similarity_scores = cosine_similarity(
    drug_pathway_matrix.values,
    luma_ref
).flatten()

# final table
drug_pathway_similarity_luma = pd.DataFrame({
    "drug": drug_pathway_matrix.index,
    "luma_pathway_enrichment_similarity": similarity_scores
})

drug_pathway_similarity_luma.head(), drug_pathway_similarity_luma.shape


(         drug  luma_pathway_enrichment_similarity
 0  CHEMBL1778                            0.102468
 1  CHEMBL1782                            0.028050
 2  CHEMBL1783                            0.125855
 3  CHEMBL1785                            0.048125
 4  CHEMBL1786                            0.013847,
 (512, 2))

In [196]:
drug_pathway_similarity_luma.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_LumA.csv",
    index=False
)

In [202]:
# Protein Family Coverag

In [206]:
import pandas as pd

# load HGNC protein family file
gene_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Gene_Protein_Family.txt",
    sep="\t"
)

# keep only approved genes
gene_family = gene_family[gene_family["Status"] == "Approved"]

# select required columns
gene_family = gene_family[
    ["Approved symbol", "Gene group name", "Gene group ID"]
].rename(
    columns={
        "Approved symbol": "gene",
        "Gene group name": "protein_family",
        "Gene group ID": "family_id"
    }
)

gene_family.head(), gene_family.shape

(       gene                         protein_family family_id
 0      A1BG  Immunoglobulin like domain containing       594
 1  A1BG-AS1                         Antisense RNAs      1987
 2      A1CF           RNA binding motif containing       725
 4       A2M           Alpha-2-macroglobulin family      2148
 5   A2M-AS1                         Antisense RNAs      1987,
 (44921, 3))

In [210]:



## ✅ FIX (ONE correct code cell)



# convert string representation to list
import ast

drug_targets["targets_ppi"] = drug_targets["targets_ppi"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# now explode
drug_targets_long = (
    drug_targets
    .explode("targets_ppi")
    .rename(columns={
        "targets_ppi": "gene",
        "target_chembl_id": "drug"
    })
)

drug_targets_long.head(), drug_targets_long.shape


(         drug    targets   gene
 0  CHEMBL1778  ['IL2RA']  IL2RA
 1  CHEMBL1782   ['FDPS']   FDPS
 2  CHEMBL1783  ['VEGFA']  VEGFA
 3  CHEMBL1785  ['EDNRB']  EDNRB
 4  CHEMBL1786  ['IMPA1']  IMPA1,
 (522, 3))

In [212]:
# load gene → protein family mapping
gene_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Gene_Protein_Family.txt",
    sep="\t"
)

gene_family.head(), gene_family.shape

(      HGNC ID Approved symbol            Status             Previous symbols  \
 0      HGNC:5            A1BG          Approved                          NaN   
 1  HGNC:37133        A1BG-AS1          Approved  NCRNA00181, A1BGAS, A1BG-AS   
 2  HGNC:24086            A1CF          Approved                          NaN   
 3      HGNC:6           A1S9T  Symbol Withdrawn                          NaN   
 4      HGNC:7             A2M          Approved                          NaN   
 
           Accession numbers                        Gene group name  \
 0                       NaN  Immunoglobulin like domain containing   
 1                  BC040926                         Antisense RNAs   
 2                  AF271790           RNA binding motif containing   
 3                       NaN                                    NaN   
 4  BX647329, X68728, M11313           Alpha-2-macroglobulin family   
 
   Gene group ID  
 0           594  
 1          1987  
 2           725  
 3      

In [214]:
# keep only required columns from HGNC
gene_family_clean = gene_family[
    ["Approved symbol", "Gene group name", "Gene group ID"]
].rename(columns={
    "Approved symbol": "gene",
    "Gene group name": "protein_family",
    "Gene group ID": "family_id"
})

# merge drug targets with protein families
drug_family = drug_targets_long.merge(
    gene_family_clean,
    on="gene",
    how="left"
)

drug_family.head(), drug_family.shape

(         drug    targets   gene  \
 0  CHEMBL1778  ['IL2RA']  IL2RA   
 1  CHEMBL1782   ['FDPS']   FDPS   
 2  CHEMBL1783  ['VEGFA']  VEGFA   
 3  CHEMBL1785  ['EDNRB']  EDNRB   
 4  CHEMBL1786  ['IMPA1']  IMPA1   
 
                                       protein_family     family_id  
 0  CD molecules|Interleukin receptors|Sushi domai...  471|602|1179  
 1                                                NaN           NaN  
 2                                        VEGF family          1267  
 3                               Endothelin receptors           225  
 4                      Phosphoinositide phosphatases          1079  ,
 (522, 5))

In [216]:
# aggregate protein family features per drug
drug_family_features = (
    drug_family
    .dropna(subset=["family_id"])
    .groupby("drug", as_index=False)
    .agg(
        n_families=("family_id", "nunique"),
        n_targets_with_family=("gene", "nunique")
    )
)

drug_family_features.head(), drug_family_features.shape

(         drug  n_families  n_targets_with_family
 0  CHEMBL1778           1                      1
 1  CHEMBL1783           1                      1
 2  CHEMBL1785           1                      1
 3  CHEMBL1786           1                      1
 4  CHEMBL1787           1                      1,
 (497, 3))

In [218]:
drug_family.groupby("drug")["family_id"].nunique().value_counts()


family_id
1    497
0     25
Name: count, dtype: int64

In [220]:
import pandas as pd

feat_luma_core = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumA.csv"
)

feat_luma_overlap = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_LumA.csv"
)

feat_luma_enrich = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_LumA.csv"
)

In [222]:
# merge sequentially on drug
luma_features_all = (
    feat_luma_core
    .merge(feat_luma_overlap, on="drug", how="left")
    .merge(feat_luma_enrich, on="drug", how="left")
)

luma_features_all.shape, luma_features_all.head()


((512, 8),
          drug  n_pathways_luma  mean_pathway_score_luma  \
 0  CHEMBL1778               16                 0.000352   
 1  CHEMBL1782                7                 0.000152   
 2  CHEMBL1783               29                 0.000318   
 3  CHEMBL1785               10                 0.000215   
 4  CHEMBL1786                3                 0.000111   
 
    max_pathway_score_luma  sum_pathway_score_luma  \
 0                0.000618                0.005637   
 1                0.000214                0.001061   
 2                0.000748                0.009235   
 3                0.000357                0.002154   
 4                0.000146                0.000333   
 
    luma_pathway_overlap_jaccard  n_shared_pathways_luma  \
 0                      0.006006                      16   
 1                      0.002628                       7   
 2                      0.010886                      29   
 3                      0.003754                      10   
 

In [224]:
luma_features_all.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumA_FINAL.csv",
    index=False
)


# Luminal B

In [62]:
import pandas as pd

# 1) load inputs
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

lumb_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_Pathway_Scores.csv"
)

# 2) create sets
drug_to_pathways = (
    drug_pathways
    .groupby("drug")["reactome_id"]
    .apply(set)
)

lumb_core_pathways = set(lumb_pathways["reactome_id"])

# 3) Jaccard overlap
overlap_records = []

for drug, d_paths in drug_to_pathways.items():
    intersection = len(d_paths & lumb_core_pathways)
    union = len(d_paths | lumb_core_pathways)
    jaccard = intersection / union if union > 0 else 0
    
    overlap_records.append({
        "drug": drug,
        "lumb_pathway_overlap_jaccard": jaccard,
        "n_shared_pathways_lumb": intersection
    })

drug_overlap_lumb = pd.DataFrame(overlap_records)

# 4) inspect
drug_overlap_lumb.shape, drug_overlap_lumb.head()

((512, 3),
          drug  lumb_pathway_overlap_jaccard  n_shared_pathways_lumb
 0  CHEMBL1778                      0.009096                      16
 1  CHEMBL1782                      0.003409                       6
 2  CHEMBL1783                      0.016487                      29
 3  CHEMBL1785                      0.005685                      10
 4  CHEMBL1786                      0.001136                       2)

In [64]:
drug_overlap_lumb.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_LumB.csv",
    index=False
)


In [66]:
import pandas as pd
import numpy as np

#  Load inputs
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

lumb_pathway_scores = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_Pathway_Scores.csv"
)

# Build LumB pathway score vector
lumb_vec = (
    lumb_pathway_scores
    .set_index("reactome_id")["lumb_pathway_score"]
)

# Compute enrichment similarity (cosine-like, weighted overlap)
records = []

for drug, df in drug_pathways.groupby("drug"):
    paths = df["reactome_id"].unique()
    
    common = lumb_vec.loc[lumb_vec.index.intersection(paths)]
    
    if len(common) == 0:
        score = 0.0
    else:
        score = common.sum() / np.sqrt((lumb_vec**2).sum())
    
    records.append({
        "drug": drug,
        "lumb_pathway_enrichment_similarity": score
    })

drug_enrich_lumb = pd.DataFrame(records)

# Inspect
drug_enrich_lumb.shape, drug_enrich_lumb.head()


((512, 2),
          drug  lumb_pathway_enrichment_similarity
 0  CHEMBL1778                            0.313792
 1  CHEMBL1782                            0.094422
 2  CHEMBL1783                            0.628607
 3  CHEMBL1785                            0.172701
 4  CHEMBL1786                            0.034852)

In [116]:
drug_enrich_lumb.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_LumB.csv",
    index=False
)

In [70]:
import pandas as pd
import ast

# 1️⃣ load drug → target mapping
drug_targets = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

# convert list strings to lists
drug_targets["targets_ppi"] = drug_targets["targets_ppi"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

drug_targets_long = (
    drug_targets
    .explode("targets_ppi")
    .rename(columns={"targets_ppi": "gene", "target_chembl_id": "drug"})
)

# 2️⃣ load HGNC gene → protein family mapping
gene_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Gene_Protein_Family.txt",
    sep="\t"
)

gene_family = gene_family[gene_family["Status"] == "Approved"][
    ["Approved symbol", "Gene group ID"]
].rename(columns={
    "Approved symbol": "gene",
    "Gene group ID": "family_id"
})

# 3️⃣ merge targets → families
drug_family = drug_targets_long.merge(
    gene_family, on="gene", how="left"
)

# 4️⃣ aggregate per drug
drug_family_features_lumb = (
    drug_family
    .dropna(subset=["family_id"])
    .groupby("drug", as_index=False)
    .agg(
        n_families_lumb=("family_id", "nunique"),
        n_targets_with_family_lumb=("gene", "nunique")
    )
)

drug_family_features_lumb.shape, drug_family_features_lumb.head()


((497, 3),
          drug  n_families_lumb  n_targets_with_family_lumb
 0  CHEMBL1778                1                           1
 1  CHEMBL1783                1                           1
 2  CHEMBL1785                1                           1
 3  CHEMBL1786                1                           1
 4  CHEMBL1787                1                           1)

In [74]:
drug_family_features_lumb.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Protein_Family_Features_LumB.csv",
    index=False
)


In [76]:
import pandas as pd

# load LumB feature blocks
feat_lumb_core = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumB.csv"
)

feat_lumb_overlap = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_LumB.csv"
)

feat_lumb_enrich = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_LumB.csv"
)

feat_lumb_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Protein_Family_Features_LumB.csv"
)

# merge all on drug
lumb_features_all = (
    feat_lumb_core
    .merge(feat_lumb_overlap, on="drug", how="left")
    .merge(feat_lumb_enrich, on="drug", how="left")
    .merge(feat_lumb_family, on="drug", how="left")
)

lumb_features_all.shape, lumb_features_all.head()


((510, 10),
          drug  n_pathways_lumb  mean_pathway_score_lumb  \
 0  CHEMBL1778               16                 0.000553   
 1  CHEMBL1782                6                 0.000444   
 2  CHEMBL1783               29                 0.000612   
 3  CHEMBL1785               10                 0.000487   
 4  CHEMBL1786                2                 0.000492   
 
    max_pathway_score_lumb  sum_pathway_score_lumb  \
 0                0.000623                0.008854   
 1                0.000529                0.002664   
 2                0.001022                0.017737   
 3                0.000611                0.004873   
 4                0.000494                0.000983   
 
    lumb_pathway_overlap_jaccard  n_shared_pathways_lumb  \
 0                      0.009096                      16   
 1                      0.003409                       6   
 2                      0.016487                      29   
 3                      0.005685                      10   


In [80]:
lumb_features_all.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_LumB_FINAL.csv",
    index=False
)


# HER

In [93]:
import pandas as pd

# load inputs
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

her2_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_Pathway_Scores.csv"
)

# create pathway sets
drug_to_pathways = (
    drug_pathways
    .groupby("drug")["reactome_id"]
    .apply(set)
)

her2_core_pathways = set(her2_pathways["reactome_id"])

# compute Jaccard overlap
records = []
for drug, d_paths in drug_to_pathways.items():
    inter = len(d_paths & her2_core_pathways)
    union = len(d_paths | her2_core_pathways)
    jaccard = inter / union if union > 0 else 0

    records.append({
        "drug": drug,
        "her2_pathway_overlap_jaccard": jaccard,
        "n_shared_pathways_her2": inter
    })

drug_overlap_her2 = pd.DataFrame(records)

drug_overlap_her2.shape, drug_overlap_her2.head()


((512, 3),
          drug  her2_pathway_overlap_jaccard  n_shared_pathways_her2
 0  CHEMBL1778                      0.009019                      16
 1  CHEMBL1782                      0.002815                       5
 2  CHEMBL1783                      0.016347                      29
 3  CHEMBL1785                      0.005637                      10
 4  CHEMBL1786                      0.001127                       2)

In [95]:
drug_overlap_her2.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_HER2.csv",
    index=False
)


In [97]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1) load inputs
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

her2_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_Pathway_Scores.csv"
)

# 2) build universe of pathways
all_pathways = sorted(
    set(drug_pathways["reactome_id"]) |
    set(her2_pathways["reactome_id"])
)

# 3) HER2 vector (pathway importance)
her2_vec = (
    her2_pathways
    .set_index("reactome_id")["her2_pathway_score"]
    .reindex(all_pathways, fill_value=0)
    .values.reshape(1, -1)
)

# 4) drug vectors + cosine similarity
records = []

for drug, df in drug_pathways.groupby("drug"):
    drug_vec = (
        df["reactome_id"]
        .value_counts()
        .reindex(all_pathways, fill_value=0)
        .values.reshape(1, -1)
    )
    
    sim = cosine_similarity(drug_vec, her2_vec)[0, 0]
    records.append({
        "drug": drug,
        "her2_pathway_enrichment_similarity": sim
    })

drug_enrich_her2 = pd.DataFrame(records)

# 5) inspect
drug_enrich_her2.shape, drug_enrich_her2.head()


((512, 2),
          drug  her2_pathway_enrichment_similarity
 0  CHEMBL1778                            0.078278
 1  CHEMBL1782                            0.030522
 2  CHEMBL1783                            0.117077
 3  CHEMBL1785                            0.054626
 4  CHEMBL1786                            0.020288)

In [99]:
drug_enrich_her2.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_HER2.csv",
    index=False
)


In [101]:
import pandas as pd
import ast

# 1) load drug → target genes (PPI-filtered)
drug_targets = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

# ensure list type
drug_targets["targets_ppi"] = drug_targets["targets_ppi"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# explode to long format
drug_targets_long = (
    drug_targets
    .explode("targets_ppi")
    .rename(columns={
        "targets_ppi": "gene",
        "target_chembl_id": "drug"
    })
)

# 2) load gene → protein family mapping (HGNC)
gene_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Gene_Protein_Family.txt",
    sep="\t"
)

# keep approved genes only
gene_family = gene_family[gene_family["Status"] == "Approved"]

gene_family = gene_family.rename(columns={
    "Approved symbol": "gene",
    "Gene group name": "protein_family",
    "Gene group ID": "family_id"
})[["gene", "protein_family", "family_id"]]

# 3) merge drug targets with protein families
drug_family = drug_targets_long.merge(
    gene_family,
    on="gene",
    how="left"
)

# 4) aggregate to drug-level features (HER2)
drug_family_features_her2 = (
    drug_family
    .groupby("drug", as_index=False)
    .agg(
        n_families_her2=("family_id", "nunique"),
        n_targets_with_family_her2=("family_id", "count")
    )
)

# 5) inspect
drug_family_features_her2.shape, drug_family_features_her2.head()


((522, 3),
          drug  n_families_her2  n_targets_with_family_her2
 0  CHEMBL1778                1                           1
 1  CHEMBL1782                0                           0
 2  CHEMBL1783                1                           1
 3  CHEMBL1785                1                           1
 4  CHEMBL1786                1                           1)

In [103]:
drug_family_features_her2.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Protein_Family_Features_HER2.csv",
    index=False
)


In [105]:
import pandas as pd

# load HER2 feature blocks
feat_her2_core = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_HER2.csv"
)

feat_her2_overlap = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_HER2.csv"
)

feat_her2_enrich = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_HER2.csv"
)

feat_her2_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Protein_Family_Features_HER2.csv"
)

# merge all on drug
her2_features_all = (
    feat_her2_core
    .merge(feat_her2_overlap, on="drug", how="left")
    .merge(feat_her2_enrich, on="drug", how="left")
    .merge(feat_her2_family, on="drug", how="left")
)

# sanity check
her2_features_all.shape, her2_features_all.head()


((510, 10),
          drug  n_pathways_her2  mean_pathway_score_her2  \
 0  CHEMBL1778               16                 0.000545   
 1  CHEMBL1782                5                 0.000450   
 2  CHEMBL1783               29                 0.000606   
 3  CHEMBL1785               10                 0.000481   
 4  CHEMBL1786                2                 0.000490   
 
    max_pathway_score_her2  sum_pathway_score_her2  \
 0                0.000627                0.008726   
 1                0.000511                0.002250   
 2                0.001017                0.017570   
 3                0.000591                0.004814   
 4                0.000497                0.000979   
 
    her2_pathway_overlap_jaccard  n_shared_pathways_her2  \
 0                      0.009019                      16   
 1                      0.002815                       5   
 2                      0.016347                      29   
 3                      0.005637                      10   


In [107]:
her2_features_all.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_HER2_FINAL.csv",
    index=False
)


# TNBC 

In [130]:
import pandas as pd

# load inputs
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

tnbc_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TNBC_Pathway_Scores.csv"
)

# build sets
drug_to_pathways = (
    drug_pathways
    .groupby("drug")["reactome_id"]
    .apply(set)
)

tnbc_core_pathways = set(tnbc_pathways["reactome_id"])

# jaccard overlap
records = []
for drug, d_paths in drug_to_pathways.items():
    inter = len(d_paths & tnbc_core_pathways)
    union = len(d_paths | tnbc_core_pathways)
    records.append({
        "drug": drug,
        "tnbc_pathway_overlap_jaccard": inter / union if union > 0 else 0,
        "n_shared_pathways_tnbc": inter
    })

drug_overlap_tnbc = pd.DataFrame(records)
drug_overlap_tnbc.shape, drug_overlap_tnbc.head()


((512, 3),
          drug  tnbc_pathway_overlap_jaccard  n_shared_pathways_tnbc
 0  CHEMBL1778                      0.009009                      16
 1  CHEMBL1782                      0.002812                       5
 2  CHEMBL1783                      0.016329                      29
 3  CHEMBL1785                      0.005631                      10
 4  CHEMBL1786                      0.001125                       2)

In [132]:
drug_overlap_tnbc.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_TNBC.csv",
    index=False
)


In [134]:
import numpy as np

tnbc_scores = (
    tnbc_pathways
    .set_index("reactome_id")["tnbc_pathway_score"]
)

records = []
for drug, d_paths in drug_to_pathways.items():
    shared = list(d_paths & set(tnbc_scores.index))
    if len(shared) == 0:
        sim = 0
    else:
        sim = np.mean(tnbc_scores.loc[shared])
    records.append({
        "drug": drug,
        "tnbc_pathway_enrichment_similarity": sim
    })

drug_enrich_tnbc = pd.DataFrame(records)
drug_enrich_tnbc.shape, drug_enrich_tnbc.head()


((512, 2),
          drug  tnbc_pathway_enrichment_similarity
 0  CHEMBL1778                            0.000544
 1  CHEMBL1782                            0.000443
 2  CHEMBL1783                            0.000600
 3  CHEMBL1785                            0.000482
 4  CHEMBL1786                            0.000479)

In [136]:
drug_enrich_tnbc.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_TNBC.csv",
    index=False
)


In [138]:
# load protein family mapping
gene_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Gene_Protein_Family.txt",
    sep="\t"
)

gene_family = gene_family[gene_family["Status"] == "Approved"][
    ["Approved symbol", "Gene group ID"]
].rename(
    columns={
        "Approved symbol": "gene",
        "Gene group ID": "family_id"
    }
)

# load drug targets
drug_targets = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

import ast
drug_targets["targets_ppi"] = drug_targets["targets_ppi"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

drug_targets_long = (
    drug_targets
    .explode("targets_ppi")
    .rename(columns={
        "target_chembl_id": "drug",
        "targets_ppi": "gene"
    })
)

# merge with family info
drug_family = drug_targets_long.merge(
    gene_family,
    on="gene",
    how="left"
)

drug_family_features_tnbc = (
    drug_family
    .groupby("drug", as_index=False)
    .agg(
        n_families_tnbc=("family_id", "nunique"),
        n_targets_with_family_tnbc=("family_id", "count")
    )
)

drug_family_features_tnbc.shape, drug_family_features_tnbc.head()


((522, 3),
          drug  n_families_tnbc  n_targets_with_family_tnbc
 0  CHEMBL1778                1                           1
 1  CHEMBL1782                0                           0
 2  CHEMBL1783                1                           1
 3  CHEMBL1785                1                           1
 4  CHEMBL1786                1                           1)

In [140]:
drug_family_features_tnbc.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Protein_Family_Features_TNBC.csv",
    index=False
)


In [142]:
# load TNBC feature blocks
feat_tnbc_core = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_TNBC.csv"
)

feat_tnbc_overlap = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Overlap_TNBC.csv"
)

feat_tnbc_enrich = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Enrichment_Similarity_TNBC.csv"
)

feat_tnbc_family = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Protein_Family_Features_TNBC.csv"
)

# merge all
tnbc_features_all = (
    feat_tnbc_core
    .merge(feat_tnbc_overlap, on="drug", how="left")
    .merge(feat_tnbc_enrich, on="drug", how="left")
    .merge(feat_tnbc_family, on="drug", how="left")
)

tnbc_features_all.shape, tnbc_features_all.head()


((510, 10),
          drug  n_pathways_tnbc  mean_pathway_score_tnbc  \
 0  CHEMBL1778               16                 0.000544   
 1  CHEMBL1782                5                 0.000443   
 2  CHEMBL1783               29                 0.000600   
 3  CHEMBL1785               10                 0.000482   
 4  CHEMBL1786                2                 0.000479   
 
    max_pathway_score_tnbc  sum_pathway_score_tnbc  \
 0                0.000618                0.008698   
 1                0.000504                0.002214   
 2                0.001012                0.017412   
 3                0.000594                0.004819   
 4                0.000479                0.000957   
 
    tnbc_pathway_overlap_jaccard  n_shared_pathways_tnbc  \
 0                      0.009009                      16   
 1                      0.002812                       5   
 2                      0.016329                      29   
 3                      0.005631                      10   


In [144]:
tnbc_features_all.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathway_Features_TNBC_FINAL.csv",
    index=False
)


# 6.5-Functional feature

In [153]:

import pandas as pd

# Load DEG tables
deg_luma = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_LumA_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

deg_lumb = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_LumB_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

deg_her2 = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_HER2_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

deg_tnbc = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_TNBC_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

deg_luma.shape, deg_lumb.shape, deg_her2.shape, deg_tnbc.shape

((870, 3), (4682, 3), (4684, 3), (4857, 3))

In [ ]:
# 

In [155]:
import pandas as pd

def build_lincs_signature(deg_df, fdr=0.05, lfc=1, top_n=250):
    up = (
        deg_df[(deg_df["log2FC"] >= lfc) & (deg_df["FDR"] < fdr)]
        .sort_values("log2FC", ascending=False)
        .head(top_n)
        .index
        .tolist()
    )
    
    down = (
        deg_df[(deg_df["log2FC"] <= -lfc) & (deg_df["FDR"] < fdr)]
        .sort_values("log2FC")
        .head(top_n)
        .index
        .tolist()
    )
    
    return up, down

In [157]:
deg_luma = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_LumA_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

luma_up, luma_down = build_lincs_signature(deg_luma)

len(luma_up), len(luma_down)


(250, 250)

In [165]:
pd.Series(luma_up).to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_LumA_UP.txt",
    index=False,
    header=False
)

pd.Series(luma_down).to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_LumA_DOWN.txt",
    index=False,
    header=False
)


In [167]:
## ✅ LINCS — Step 3: Repeat Disease Signatures for Other Subtypes


import pandas as pd

# helper already defined earlier:
# build_lincs_signature(deg_df, fdr=0.05, lfc=1, top_n=250)

subtypes = {
    "LumB": r"C:\Users\deep8\breast_cancer_project_folder\DEGs_LumB_vs_Normal_FDR0.05_log2FC1.csv",
    "HER2": r"C:\Users\deep8\breast_cancer_project_folder\DEGs_HER2_vs_Normal_FDR0.05_log2FC1.csv",
    "TNBC": r"C:\Users\deep8\breast_cancer_project_folder\DEGs_TNBC_vs_Normal_FDR0.05_log2FC1.csv",
}

results = {}

for subtype, path in subtypes.items():
    deg = pd.read_csv(path, index_col=0)
    up, down = build_lincs_signature(deg)
    
    results[subtype] = (len(up), len(down))
    
    pd.Series(up).to_csv(
        fr"C:\Users\deep8\breast_cancer_project_folder\LINCS_{subtype}_UP.txt",
        index=False, header=False
    )
    pd.Series(down).to_csv(
        fr"C:\Users\deep8\breast_cancer_project_folder\LINCS_{subtype}_DOWN.txt",
        index=False, header=False
    )

results



{'LumB': (250, 250), 'HER2': (250, 250), 'TNBC': (250, 250)}

In [177]:

import pandas as pd

lincs = pd.read_csv(
    r"C:\Users\deep8\Downloads\Project data\LINCS_level5_subset_with_symbols.tsv",
    sep="\t"
)

lincs.shape, lincs.head()

((1088, 12329),
                                         cid     PSME1      ATF1      RHEB  \
 0  CPC004_A375_6H:BRD-K39120595-304-03-9:10  0.842056  1.851618  0.223348   
 1  CPC004_A375_6H:BRD-K02637541-001-06-5:10 -2.125443  1.558020  1.657636   
 2  CPC004_A375_6H:BRD-A52530684-001-01-1:10  1.950261 -1.775065  0.738742   
 3  CPC005_A375_6H:BRD-A76941896-003-02-0:10  4.170150 -1.709900 -0.316700   
 4  CPC005_A375_6H:BRD-K59637651-001-01-4:10  0.197183 -0.444997 -0.197916   
 
       FOXO3      RHOA      IL1B     ASAH1      RALA  ARHGEF12  ...     GSK3A  \
 0  0.776297  1.704985  1.661618  1.578550  2.780362  1.407401  ...  1.868213   
 1  1.717058  0.526033  0.507814  0.931818  0.857882  0.446190  ...  2.403398   
 2 -2.392857  1.393126  2.806955  1.421755 -1.072971  1.018446  ...  1.889073   
 3 -3.955550  3.753400  3.964850  2.775750  0.079450  2.252550  ...  3.204750   
 4 -0.234726 -0.340859  0.475621 -0.079726  0.053035 -1.051894  ...  0.319730   
 
      PKNOX2  MAPKAPK5-AS1

In [179]:
# extract drug name from cid
lincs["drug"] = lincs["cid"].str.split(":").str[1]

# drop cid
lincs_mat = lincs.drop(columns=["cid"])

# average signatures per drug
lincs_drug = (
    lincs_mat
    .groupby("drug")
    .mean()
)

lincs_drug.shape, lincs_drug.head()

((44, 12328),
                            PSME1      ATF1      RHEB     FOXO3      RHOA  \
 drug                                                                       
 BRD-A23723433-001-01-2 -0.172919  0.073394 -0.029062  0.019874 -0.208624   
 BRD-A23723433-001-02-0 -0.328317 -0.135014  0.596025 -0.719783 -1.188171   
 BRD-A28746609-001-05-7 -0.336155 -0.467016  0.151469 -0.305589 -0.646775   
 BRD-A52530684-001-01-1  1.177006 -2.123511 -1.068096 -4.187952  1.018609   
 BRD-A52530684-003-01-7 -0.281655 -0.280919 -2.481116 -0.106861 -1.223447   
 
                             IL1B     ASAH1      RALA  ARHGEF12      SOX2  ...  \
 drug                                                                      ...   
 BRD-A23723433-001-01-2  0.158527 -0.030645 -0.126778 -0.081933 -0.045822  ...   
 BRD-A23723433-001-02-0  1.277178  0.430860  0.768792  0.155072  0.669201  ...   
 BRD-A28746609-001-05-7  0.125174 -0.154901  0.094219 -0.075582  0.082050  ...   
 BRD-A52530684-001-01-1  2.734370  

In [183]:
# save for reuse
lincs_drug.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_Drug_Signatures.csv"
)


In [193]:
import numpy as np

def lincs_connectivity(drug_signature, up_genes, down_genes):
    # keep genes present in LINCS
    up = list(set(up_genes) & set(drug_signature.index))
    down = list(set(down_genes) & set(drug_signature.index))
    
    if len(up) == 0 or len(down) == 0:
        return np.nan
    
    # LINCS-style connectivity (mean difference)
    score = drug_signature[down].mean() - drug_signature[up].mean()
    return score


In [195]:
# LINCS — Step 3: Compute Luminal A connectivity scores (ONE cell)

import pandas as pd
import numpy as np

# load drug signatures
lincs_drug = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_Drug_Signatures.csv",
    index_col=0
)

# load Luminal A disease signature
luma_up = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_LumA_UP.txt",
    header=None
)[0].tolist()

luma_down = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_LumA_DOWN.txt",
    header=None
)[0].tolist()

# connectivity function (already defined earlier)
def lincs_connectivity(drug_signature, up_genes, down_genes):
    up = list(set(up_genes) & set(drug_signature.index))
    down = list(set(down_genes) & set(drug_signature.index))
    if len(up) == 0 or len(down) == 0:
        return np.nan
    return drug_signature[down].mean() - drug_signature[up].mean()

# compute connectivity per drug
records = []
for drug, sig in lincs_drug.iterrows():
    score = lincs_connectivity(sig, luma_up, luma_down)
    records.append({"drug": drug, "lincs_connectivity_luma": score})

lincs_luma = pd.DataFrame(records)

lincs_luma.shape, lincs_luma.head()


((44, 2),
                      drug  lincs_connectivity_luma
 0  BRD-A23723433-001-01-2                 0.027447
 1  BRD-A23723433-001-02-0                -0.085848
 2  BRD-A28746609-001-05-7                -0.049871
 3  BRD-A52530684-001-01-1                -0.255084
 4  BRD-A52530684-003-01-7                -0.217272)

In [197]:


lincs_luma.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LINCS_Connectivity_LumA.csv",
    index=False
)

print("✅ Saved: Drug_LINCS_Connectivity_LumA.csv")


✅ Saved: Drug_LINCS_Connectivity_LumA.csv


In [201]:

import pandas as pd
import numpy as np

# load averaged drug signatures
lincs_drug = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_Drug_Signatures.csv",
    index_col=0
)

# load LumB signatures
lumb_up = set(pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_LumB_UP.txt",
    header=None
)[0])

lumb_down = set(pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_LumB_DOWN.txt",
    header=None
)[0])

# intersect genes properly
up_genes = list(lumb_up & set(lincs_drug.columns))
down_genes = list(lumb_down & set(lincs_drug.columns))

records = []
for drug, row in lincs_drug.iterrows():
    up_score = row[up_genes].mean()
    down_score = row[down_genes].mean()
    connectivity = down_score - up_score
    records.append({
        "drug": drug,
        "lincs_connectivity_lumb": connectivity
    })

lincs_lumb = pd.DataFrame(records)

lincs_lumb.shape, lincs_lumb.head()


((44, 2),
                      drug  lincs_connectivity_lumb
 0  BRD-A23723433-001-01-2                -0.096401
 1  BRD-A23723433-001-02-0                 0.243366
 2  BRD-A28746609-001-05-7                -0.099070
 3  BRD-A52530684-001-01-1                 0.895618
 4  BRD-A52530684-003-01-7                 0.866149)

In [203]:
# save Luminal B LINCS connectivity
lincs_lumb.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_Connectivity_LumB.csv",
    index=False
)

print("✅ Saved: LINCS_Connectivity_LumB.csv")


✅ Saved: LINCS_Connectivity_LumB.csv


In [207]:
import pandas as pd

# load HER2 LINCS signatures you already saved
her2_up = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_HER2_UP.txt",
    header=None
)[0].tolist()

her2_down = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_HER2_DOWN.txt",
    header=None
)[0].tolist()

len(her2_up), len(her2_down)


(250, 250)

In [209]:
import numpy as np
import pandas as pd

# genes available in LINCS
lincs_genes = set(lincs_drug.columns)

# intersect DEG signatures with LINCS genes
her2_up_lincs = list(set(her2_up) & lincs_genes)
her2_down_lincs = list(set(her2_down) & lincs_genes)

records = []
for drug, row in lincs_drug.iterrows():
    up_score = row[her2_up_lincs].mean()
    down_score = row[her2_down_lincs].mean()
    connectivity = down_score - up_score

    records.append({
        "drug": drug,
        "lincs_connectivity_her2": connectivity
    })

lincs_her2 = pd.DataFrame(records)

# inspect
lincs_her2.shape, lincs_her2.head()


((44, 2),
                      drug  lincs_connectivity_her2
 0  BRD-A23723433-001-01-2                -0.086310
 1  BRD-A23723433-001-02-0                 0.219983
 2  BRD-A28746609-001-05-7                -0.082671
 3  BRD-A52530684-001-01-1                 0.996127
 4  BRD-A52530684-003-01-7                 0.844745)

In [211]:
lincs_her2.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_LINCS_Connectivity_HER2.csv",
    index=False
)


In [223]:
# build TNBC up/down lists again
import pandas as pd

deg_tnbc = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_TNBC_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

# standard LINCS choice
tnbc_up = (
    deg_tnbc
    .query("log2FC > 1")
    .sort_values("log2FC", ascending=False)
    .head(250)
    .index
    .tolist()
)

tnbc_down = (
    deg_tnbc
    .query("log2FC < -1")
    .sort_values("log2FC")
    .head(250)
    .index
    .tolist()
)

len(tnbc_up), len(tnbc_down)


(250, 250)

In [225]:
# intersect TNBC DEG signatures with LINCS genes
lincs_genes = set(lincs_drug.columns)

tnbc_up_lincs = list(set(tnbc_up) & lincs_genes)
tnbc_down_lincs = list(set(tnbc_down) & lincs_genes)

len(tnbc_up_lincs), len(tnbc_down_lincs)


(186, 169)

In [227]:
# percentage retained
len(tnbc_up_lincs) / 250, len(tnbc_down_lincs) / 250


(0.744, 0.676)

In [229]:
import pandas as pd
import numpy as np

# LINCS drug signatures (already computed)
lincs_drug = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_Drug_Signatures.csv",
    index_col=0
)

# load TNBC LINCS gene lists
tnbc_up = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_TNBC_UP.txt",
    header=None
)[0].tolist()

tnbc_down = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_TNBC_DOWN.txt",
    header=None
)[0].tolist()

# intersect with LINCS genes
lincs_genes = set(lincs_drug.columns)
tnbc_up_lincs = list(set(tnbc_up) & lincs_genes)
tnbc_down_lincs = list(set(tnbc_down) & lincs_genes)

# compute connectivity
records = []
for drug, row in lincs_drug.iterrows():
    up_score = row[tnbc_up_lincs].mean()
    down_score = row[tnbc_down_lincs].mean()
    connectivity = down_score - up_score
    
    records.append({
        "drug": drug,
        "lincs_connectivity_tnbc": connectivity
    })

lincs_tnbc = pd.DataFrame(records)

lincs_tnbc.shape, lincs_tnbc.head()


((44, 2),
                      drug  lincs_connectivity_tnbc
 0  BRD-A23723433-001-01-2                -0.105868
 1  BRD-A23723433-001-02-0                 0.227465
 2  BRD-A28746609-001-05-7                -0.081344
 3  BRD-A52530684-001-01-1                 1.193621
 4  BRD-A52530684-003-01-7                 0.900220)

In [231]:
lincs_tnbc.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LINCS_Connectivity_TNBC.csv",
    index=False
)


# PHASE 6.5.2 — CRISPR Essentiality Correlation

In [236]:
import pandas as pd

# DepMap CRISPR gene effect (example filename)
crispr = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\CRISPR_gene_effect.csv",
    index_col=0
)

crispr.shape, crispr.iloc[:5, :5]


((1086, 17386),
             A1BG (1)  A1CF (29974)   A2M (2)  A2ML1 (144568)  A3GALT2 (127550)
 DepMap_ID                                                                     
 ACH-000001 -0.134808      0.059764 -0.008665       -0.003572         -0.106211
 ACH-000004  0.081853     -0.056401 -0.106738       -0.014499          0.078209
 ACH-000005 -0.094196     -0.014598  0.100426        0.169103          0.032363
 ACH-000007 -0.011544     -0.123189  0.080692        0.061046         -0.013454
 ACH-000009 -0.050782     -0.037466  0.068885        0.090375          0.012634)

In [250]:


import pandas as pd

sample_info = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\sample_info.csv"
)

sample_info.columns


Index(['DepMap_ID', 'cell_line_name', 'stripped_cell_line_name', 'CCLE_Name',
       'alias', 'COSMICID', 'sex', 'source', 'RRID', 'WTSI_Master_Cell_ID',
       'sample_collection_site', 'primary_or_metastasis', 'primary_disease',
       'Subtype', 'age', 'Sanger_Model_ID', 'depmap_public_comments',
       'lineage', 'lineage_subtype', 'lineage_sub_subtype',
       'lineage_molecular_subtype', 'default_growth_pattern',
       'model_manipulation', 'model_manipulation_details', 'patient_id',
       'parent_depmap_id', 'Cellosaurus_NCIt_disease', 'Cellosaurus_NCIt_id',
       'Cellosaurus_issues'],
      dtype='object')

In [278]:
drug_targets.columns

Index(['target_chembl_id', 'targets', 'targets_ppi'], dtype='object')

In [280]:

import ast
import pandas as pd

# make a working copy
drug_targets_fix = drug_targets.copy()

# ensure targets_ppi is a list
drug_targets_fix["targets_ppi"] = drug_targets_fix["targets_ppi"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# explode to long format
drug_targets_long = (
    drug_targets_fix
    .explode("targets_ppi")
    .rename(columns={
        "target_chembl_id": "drug",
        "targets_ppi": "gene"
    })
    [["drug", "gene"]]
    .dropna()
)

drug_targets_long.head(), drug_targets_long.shape


(         drug   gene
 0  CHEMBL1778  IL2RA
 1  CHEMBL1782   FDPS
 2  CHEMBL1783  VEGFA
 3  CHEMBL1785  EDNRB
 4  CHEMBL1786  IMPA1,
 (522, 2))

In [282]:
import numpy as np
import pandas as pd

# 1) Load CRISPR gene effect (Chronos-corrected)
crispr = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\CRISPR_gene_effect.csv",
    index_col=0
)

# 2) Load DepMap sample metadata
sample_info = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\sample_info.csv"
)

# 3) Keep only breast cancer cell lines
breast_cells = sample_info[
    sample_info["primary_disease"].str.contains("Breast", case=False, na=False)
]

# match CRISPR rows
crispr_breast = crispr.loc[
    crispr.index.intersection(breast_cells["DepMap_ID"])
]

crispr_breast.shape

(47, 17386)

In [286]:
# FIX CRISPR column names: "TP53 (7157)" → "TP53"
crispr_breast_clean = crispr_breast.copy()
crispr_breast_clean.columns = (
    crispr_breast_clean.columns
    .str.split(" ")
    .str[0]
)

# sanity check
crispr_breast_clean.columns[:5]


Index(['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2'], dtype='object')

In [288]:
records = []

for drug, df in drug_targets_long.groupby("drug"):
    genes = list(set(df["gene"]) & set(crispr_breast_clean.columns))
    if len(genes) == 0:
        continue

    score = crispr_breast_clean[genes].mean(axis=1).mean()

    records.append({
        "drug": drug,
        "crispr_essentiality_score": score,
        "n_targets_in_crispr": len(genes)
    })

drug_crispr_features = pd.DataFrame(records)

drug_crispr_features.shape, drug_crispr_features.head()


((508, 3),
          drug  crispr_essentiality_score  n_targets_in_crispr
 0  CHEMBL1778                  -0.009925                    1
 1  CHEMBL1782                  -0.650429                    1
 2  CHEMBL1785                   0.049749                    1
 3  CHEMBL1786                  -0.002494                    1
 4  CHEMBL1787                   0.067292                    1)

In [290]:
drug_crispr_features.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_CRISPR_Essentiality_Global.csv",
    index=False
)

#  CRISPR Essentiality (Subtype-specific)